In [1]:
def validate_numbers(repository):
    stars = repository['stars']
    forks = repository['forks']
    watchers = repository['watchers']
    languageCount = repository['languageCount']

    if (stars < 0 or forks < 0 or watchers < 0 or languageCount < 0):
        return False
    return True

In [2]:
from datetime import date
def validate_date(repository):
    createdAt = repository['createdAt']
    pushedAt = repository['pushedAt']

    if not isinstance(createdAt,date):
        return False
    if not isinstance(pushedAt,date):
        return False
    if repository['repo_age_days'] < 0:
        return False
    return True

In [3]:
def validate_name(repository):
    name = repository['name']
    if pd.isna(name):
        return False
    if not isinstance(name,str):
        return False
    if name.strip() == "":
        return False
    return True

In [4]:
#Function 1
import pandas as pd
def load_data(file_path):
    return pd.read_csv(file_path)

In [5]:
#Function 2
def convert_dates(df):
    df['pushedAt'] = pd.to_datetime(df['pushedAt'], utc=True)
    df['createdAt'] = pd.to_datetime(df['createdAt'], utc=True)
    return df

In [6]:
def remove_empty_names(df):
    df = df[df['name'].fillna('').str.strip() != '']
    return df

In [7]:
def remove_duplicates(df):
    df = df.drop_duplicates(subset=['owner', 'name'])
    return df

In [8]:
def calculate_repo_age(df):
    df['repo_age_days'] = (
        df['pushedAt'] - df['createdAt']
    ).dt.days
    return df

In [9]:
def remove_invalid_repo_age(df):
    df = df[df['repo_age_days'] >= 0]
    return df

In [10]:
import ast

def combine_text(df):
    def combine_row(row):
        description = row['description'] if pd.notna(row['description']) else ''
        primary_language = row['primaryLanguage'] if pd.notna(row['primaryLanguage']) else ''

        languages = ast.literal_eval(row['languages']) if row['languages'] else []
        topics = ast.literal_eval(row['topics']) if row['topics'] else []

        return ' '.join([
            str(row['name']),
            str(description),
            str(primary_language),
            *languages,
            *topics
        ])

    df['combined_text'] = df.apply(combine_row, axis=1)

    return df


In [12]:
def process_data(file_path):
    df = load_data(file_path)
    df = convert_dates(df)
    df = calculate_repo_age(df)
    df = remove_invalid_repo_age(df)
    df = remove_empty_names(df)
    df = remove_duplicates(df)
    df = combine_text(df)

    return df

In [19]:
for i in range(1,43):
    input_path = f'../data/processed_v2/repositories_{i:02}.csv'
    # print(input_path)
    df = process_data(input_path)
    output_path = f'../data/processed_v3/repositories_{i:02}.csv'
    print(output_path)
    df.to_csv(output_path,index=False)
print("Finished")

../data/processed_v3/repositories_01.csv
../data/processed_v3/repositories_02.csv
../data/processed_v3/repositories_03.csv
../data/processed_v3/repositories_04.csv
../data/processed_v3/repositories_05.csv
../data/processed_v3/repositories_06.csv
../data/processed_v3/repositories_07.csv
../data/processed_v3/repositories_08.csv
../data/processed_v3/repositories_09.csv
../data/processed_v3/repositories_10.csv
../data/processed_v3/repositories_11.csv
../data/processed_v3/repositories_12.csv
../data/processed_v3/repositories_13.csv
../data/processed_v3/repositories_14.csv
../data/processed_v3/repositories_15.csv
../data/processed_v3/repositories_16.csv
../data/processed_v3/repositories_17.csv
../data/processed_v3/repositories_18.csv
../data/processed_v3/repositories_19.csv
../data/processed_v3/repositories_20.csv
../data/processed_v3/repositories_21.csv
../data/processed_v3/repositories_22.csv
../data/processed_v3/repositories_23.csv
../data/processed_v3/repositories_24.csv
../data/processe

In [21]:
df_new = pd.read_csv("../data/processed_v3/repositories_01.csv")
df_new.skew(numeric_only=True)

stars            19.131788
forks            41.114001
watchers         17.552295
languageCount     7.134400
isFork            0.000000
isArchived        3.215779
repo_age_days     0.437353
dtype: float64

In [25]:
import pandas as pd
import glob

files = glob.glob('../data/processed_v3/repositories_*.csv')

df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)
df.to_csv('../data/merged/merged.csv',index=False)